## Optimizer Comparision - Performance Tests

In [ ]:
! pip install -q neptune

### CIFAR100 Image Classification

In [3]:
from typing import Tuple

import torch
import torchvision
from torch import nn
from torch import optim
from torch.utils import data
from torchvision import models, transforms, datasets

CIFAR100_ROOT = "/data/cifar100"


def cifar100_model_factory(num_classes: int = 100) -> nn.Module:
    model = models.resnet18(num_classes=num_classes)
    # modify first conv layer to avoid upscaling to 224x224
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model

def cifar100_dataloader_factory(
    batch_size: int = 32,
    seed: int = 42
) -> Tuple[data.DataLoader, data.DataLoader, data.DataLoader]:
    # standard CIFAR-100 mean and std
    mean = [0.5071, 0.4865, 0.4409]
    std = [0.2673, 0.2564, 0.2761]

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])

    temp_dataset = datasets.CIFAR100(root=CIFAR100_ROOT, train=True, transform=transform, download=True)
    test_dataset = datasets.CIFAR100(root=CIFAR100_ROOT, train=False, transform=transform, download=True)

    train_size = int(0.8 * len(temp_dataset))
    val_size = len(temp_dataset) - train_size
    generator = torch.Generator().manual_seed(seed)
    train_dataset, val_dataset = data.random_split(temp_dataset, [train_size, val_size], generator=generator)

    train_loader = data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

In [ ]:
from datetime import datetime

import neptune
from sklearn.metrics import accuracy_score, f1_score

def cifar100_train_eval_loop(
    *,
    model: nn.Module,
    optimizer: optim.Optimizer,
    train_loader: data.DataLoader,
    val_loader: data.DataLoader,
    criterion: nn.Module = nn.CrossEntropyLoss(),
    epochs: int = 10,
    run: neptune.Run | None = None,
    device: str = "cuda",
) -> float:
    model = model.to(device)

    start_time = datetime.now()
    for epoch in range(epochs):
        if run:
            run["train/epoch/lr"].append(optimizer.param_groups[0]['lr'])

        model.train()
        train_preds = []
        train_targets = []
        train_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            train_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            train_preds.extend(preds.cpu().tolist())
            train_targets.extend(targets.cpu().tolist())

            loss.backward()
            optimizer.step()

            if run:
                run["train/step/loss"].append(loss.item())

        train_loss /= len(train_loader)
        train_f1 = f1_score(train_targets, train_preds, average="macro")
        train_accuracy = accuracy_score(train_targets, train_preds)
        if run:
            run["train/epoch/loss"].append(train_loss)
            run["train/epoch/f1"].append(train_f1)
            run["train/epoch/accuracy"].append(train_accuracy)

        model.eval()
        val_preds = []
        val_targets = []
        val_loss = 0.0
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item()

                preds = torch.argmax(outputs, dim=1)
                val_preds.extend(preds.cpu().tolist())
                val_targets.extend(targets.cpu().tolist())

                if run:
                    run["val/step/loss"].append(loss.item())

        val_loss /= len(val_loader)
        val_f1 = f1_score(val_targets, val_preds, average="macro")
        val_accuracy = accuracy_score(val_targets, val_preds)
        if run:
            run["val/epoch/loss"].append(val_loss)
            run["val/epoch/f1"].append(val_f1)
            run["val/epoch/accuracy"].append(val_accuracy)


        print(f"Epoch {epoch+1}/{epochs} — F1 Score: {val_f1:.4f} | Accuracy: {val_accuracy:.4f} | Val Loss: {val_loss:.4f} | Train Loss | {train_loss:.4f}")

    end_time = datetime.now()
    train_time = end_time - start_time
    print(f"Training time: {end_time - start_time}")
    if run:
        run["train/time"].append(train_time.total_seconds())
